# Chapitre 10 - Cours 3 : interpréter un modèle boosté (spam et données immobilières simulées)

Ce notebook illustre les sections 10.7, 10.8, 10.13 et 10.14 du livre :

1. pourquoi les arbres boostés sont une méthode « clé en main » pour la fouille de données ;
2. le gradient boosting sur les données **spam** (section 10.8) ;
3. l'**importance relative** des variables (éq. (10.42)–(10.43), figure 10.6) ;
4. les **graphes de dépendance partielle** (éq. (10.47)–(10.49), figures 10.7 et 10.8) ;
5. une étude complète sur des **données immobilières simulées**, construites sur le
   modèle des données *California Housing* de la section 10.14.1 (non disponibles ici).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
set_style()

from esl import datasets
from esl.ch10_boosting import (GradientBoosting, RegressionTree, SPAM_NAMES,
                               partial_dependence_bruteforce)

## 1. Des méthodes « clé en main » pour la fouille de données

Les données réelles de la fouille de données (*data mining*) sont souvent volumineuses,
hétérogènes (variables quantitatives et qualitatives), avec des valeurs manquantes, des
distributions très asymétriques, des valeurs aberrantes et beaucoup de variables
inutiles. Le tableau 10.1 du livre compare les méthodes selon ces critères.

Les **arbres de décision** cochent presque toutes les cases : rapides, interprétables
(s'ils sont petits), invariants par transformation monotone des prédicteurs (donc
insensibles aux échelles et aux valeurs extrêmes des $X_j$), sélection de variables
intégrée. Leur défaut est leur **imprécision**. Le boosting améliore fortement la
précision tout en conservant la plupart de ces qualités ; on perd en revanche
l'interprétabilité immédiate, d'où les outils de la section 10.13.

## 2. Les données spam

4601 messages électroniques, dont 39,4 % de spams ; les 57 prédicteurs sont des fréquences
de mots (`george`, `hp`, `remove`, `free`…) et de caractères (`!`, `$`…) ainsi que trois
statistiques sur les suites de majuscules (`CAPAVE`, `CAPMAX`, `CAPTOT`). Le livre utilise une
séparation apprentissage/test précise (1536 messages de test) que nous n'avons pas : nous
tirons au hasard 1536 messages de test.

In [2]:
spam = datasets.load_spam()
Xs = spam.drop(columns="spam").to_numpy(float)
ys = spam["spam"].to_numpy()
rng = np.random.RandomState(0)
perm = rng.permutation(len(ys))
test_idx, train_idx = perm[:1536], perm[1536:]
Xs_tr, ys_tr = Xs[train_idx], ys[train_idx]
Xs_te, ys_te = Xs[test_idx], ys[test_idx]
print("apprentissage :", Xs_tr.shape, "| test :", Xs_te.shape, "| proportion de spams : %.3f" % ys.mean())

apprentissage : (3065, 57) | test : (1536, 57) | proportion de spams : 0.394


On modélise le **log-rapport de cotes**

$$f(x) = \log\frac{P(\text{spam}\mid x)}{P(\text{email}\mid x)} \qquad \text{(éq. (10.24))}$$

par gradient boosting avec la déviance binomiale, des arbres à $J = 5$ feuilles, un
rétrécissement $\nu = 0{,}1$ et un sous-échantillonnage $\eta = 0{,}5$.

In [ ]:
M_spam = 1000
gb5 = GradientBoosting("deviance", M_spam, 5, learning_rate=0.1, subsample=0.5,
                       random_state=0).fit(Xs_tr, ys_tr)
its5, dev5, err5 = gb5.staged_scores(Xs_te, ys_te, every=10)
gb2 = GradientBoosting("deviance", M_spam, 2, learning_rate=0.1, subsample=0.5,
                       random_state=0).fit(Xs_tr, ys_tr)
its2, dev2, err2 = gb2.staged_scores(Xs_te, ys_te, every=10)